## เริ่ม Model

## ใช้ Dataset ใน Sklearn มีหลายแบบโหลดไม่เหมือนกัน

## กรณีงานจริง กำหนด Feature และ Target

In [ ]:
# กำหนด features และ target
# X = df[["column1", "column2", "column3", "column4"]]  # features
# y = df["ผลลัพธ์"]    # target
# กำหนด target_names เอง
# target_names = ["cat", "dog", "rabbit"]

## เริ่ม dataset ตัวอย่าง

In [23]:
## Forest Covertypes dataset โหลดและแปลงกลับเป็น column ต้นทางที่ยังไม่ได้ one hot encoder
## Target classes (7 ชนิดของป่า)   581012 rows × 14 columns
## covertype คือ target
# Spruce/Fir ป่าสนสปรูซและเฟอร์ 
# Lodgepole Pine ป่าสน Lodgepole 
# Ponderosa Pine ป่าสน Ponderosa 
# Cottonwood/Willow ป่าต้น Cottonwood และ Willow 
# Aspen ป่า Aspen 
# Douglas-fir ป่าสน Douglas-f 
# Krummholz ป่าพุ่มไม้เตี้ยที่ขึ้นในพื้นที่สูงมาก 


In [ ]:
import pandas as pd
forest = pd.read_csv(r'D:\Forest Covertypes.csv')
forest = forest.set_index('Index')

# forest.info()
forest.head(5)
# forest.index
# forest.columns
# forest.shape


## สำรวจข้อมูลก่อนเข้า Model เพิ่มส่วน EDA อีกครั้ง

In [ ]:
# จัดการค่า Missing
# จัดการค่า Outlier
# จัดการค่า Dupplicate

forest.info()
# forest.isnull().sum()


## แยก Train กับ Test ที่ขั้นตอนนี้เลย

In [86]:
from sklearn.model_selection import train_test_split

# แยก features และ target
X = forest.drop(columns=["Cover_Type"])   # ลบ target ออก
y = forest["Cover_Type"]                  # target

# แยก Train/Test 70:30  stratify=y รักษาสัดส่วน class ให้ข้อมูลแต่ละ Class เท่าต้นฉบับ
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,stratify=y)

# X_train
# y_train


## ต่อด้วย One hot Encoding หรือ Label Encoding

In [89]:
## จัดการ column กลุ่ม Category  ด้วย One hot Encoding pd.get_dummies(trainset,columns = catgorycols)
## ทำทั้ง Train set และ Test set
## ใช้ One hot Encoding เป็นมาตรฐาน ถึงจะเป็น label ที่มีลำดับหรือไม่มีลำดับก็ใช้ได้ดี
## ใช้ Label Encoding กับ Label ที่มีลำดับเท่านั้น
## text columns ที่เป็น free text เช่น ชื่อ-นามสกุล, หมู่บ้าน, อาคาร → ตัดออก
## ส่วนใหญ่ไม่มีความหมายเชิง category ที่ช่วยโมเดล และจำนวน unique values มักเยอะมากจนเกินเหตุตัดออก
## one-hot ไม่ practical → ตัดออกไปได้เลย

import pandas as pd

# สมมติว่ามี categorical columns
cat_cols = ["Wilderness_Area", "Soil_Type"]

# Train set ใช้ OneHotEncoder จาก Pandas
X_train_final = pd.get_dummies(X_train, columns=cat_cols)


# Test set ใช้ OneHotEncoder จาก Pandas
X_test_final  = pd.get_dummies(X_test, columns=cat_cols)


X_trainTreebase = X_train_final
X_testTreebase = X_test_final

# target y ทั้ง Train และ Test encode ตามตัวเลข
ytrainall = pd.DataFrame(y_train.astype('category').cat.codes,columns =["Cover_Type"])
ytestall = pd.DataFrame(y_test.astype('category').cat.codes,columns=["Cover_Type"])

# Feature Scaling

In [90]:
# จัดการกลุ่ม Column ตัวเลข ที่อาจมีฐานข้อมูล หน่วย ไม่เท่ากัน
# ****เราทำ Feature Scaling เพื่อปรับหน่วยตัวเลขแต่ละ Column ให้ใกล้เคียงกัน****
#    ****กรณีหน่วยมันเป็นตัวเดียวกันอยู่แล้วไม่ต้องทำตีความยาก****

# standard normailize  เหมาะกับ Logistic Regression(Linear Classification) , SVM (Distance-based ใช้ระยะทาง)
# Min-Max normailize   เหมาะกับ Neural Network, KNN (Distance-based ใช้ระยะทาง Euclidean)
# RobustScaler normailize  เหมาะกับ Dataset ที่มี outlier เยอะๆ
# Treebase Model ไม่ต้อง feature Scaling Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost
## *****Tree-based models (Random Forest, XGBoost) **** → ไม่จำเป็นต้องทำ scaling เพราะไม่อ่อนไหวต่อสเกล

from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler

# 10 column ที่เป็นตัวเลข Float มาปรับ Scale กัน
numeric_cols = ["Elevation", "Aspect", "Slope",
                "Horizontal_Distance_To_Hydrology",
                "Vertical_Distance_To_Hydrology",
                "Horizontal_Distance_To_Roadways",
                "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
                "Horizontal_Distance_To_Fire_Points"]

X_numtrain = X_train_final[numeric_cols]
X_catgorytrain = X_train_final.drop(columns = numeric_cols)

X_numtest = X_test_final[numeric_cols]
X_catgorytest = X_test_final.drop(columns = numeric_cols)


# X_catgory.columns
# X_num.columns
## แบบที่ 1 # Standardize numeric features
# scaler = StandardScaler()
# X_num_scaled = scaler.fit_transform(X_num)

## แบบที่ 2 # MinMaxScaler numeric features
# scaler = MinMaxScaler()
# X_num_scaled = scaler.fit_transform(X_num)

## แบบที่ 3 # RobustScaler  numeric features
# scaler = RobustScaler()
# X_num_scaled = scaler.fit_transform(X_num)

## ค่า x และ y
# X = pd.concat([pd.DataFrame(X_num_scaled, columns=numeric_cols), X_cat], axis=1)
# y = df["Cover_Type"]


## แยก 2 แบบก่อน RobustScaler ใช้กรณี outlier เยอะๆ

In [ ]:
## แบบที่ 1  StandardScaler  ใช้กับ Logistic Regression, SVM

from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
scaler1 = StandardScaler()            # สร้างครั้งเดียวพอ
X_num_trainscaled1 = scaler1.fit_transform(X_numtrain)



In [ ]:
## แบบที่ 1  StandardScaler บน Train set  .fit_transform
# เอา X_num_scaled ที่แปลงแล้ว เข้า dataframe และรวมกับ X_catgory ที่ไม่รวม Cover_Type
#  index=X_num.index เป็นการบอกว่า Scaler ที่แปลงแล้วให้ใช้ index เดิม

X_num_trainscaled_df = pd.DataFrame(X_num_trainscaled1, columns=numeric_cols,index=X_numtrain.index)
xtrainscaler1 = pd.concat([X_num_trainscaled_df, X_catgorytrain], axis=1)

## ได้ที่จะใช้
# xtrainscaler1

In [92]:
## แบบที่ 1  StandardScaler บน Test set  .transform
X_num_testscaled1 = scaler.transform(X_numtest)

X_num_testscaled_df = pd.DataFrame(X_num_testscaled1, columns=numeric_cols,index=X_numtest.index)
xtestscaler1 = pd.concat([X_num_testscaled_df, X_catgorytest], axis=1)


## ได้ที่จะใช้
# xtestscaler1
# ytestscaler1


In [ ]:
## แบบที่ 2 Neural Network, KNN
## แบบที่ 2  บน Train set  .fit_transform

scaler2 = MinMaxScaler()
X_num_trainscaled = scaler2.fit_transform(X_numtrain)

# เอา X_num_scaled ที่แปลงแล้ว เข้า dataframe และรวมกับ X_catgory ที่ไม่รวม Cover_Type
#  index=X_num.index เป็นการบอกว่า Scaler ที่แปลงแล้วให้ใช้ index เดิม

X_num_trainscaled_df = pd.DataFrame(X_num_trainscaled, columns=numeric_cols,index=X_num.index)
xtrainscaler1 = pd.concat([X_num_trainscaled_df, X_catgorytrain], axis=1)

# target y
ytrainscaler1 = y_train

## บน Train set  .transform
scaler2 = MinMaxScaler()
X_num_testscaled = scaler2.transform(X_numtest)

X_num_testscaled_df = pd.DataFrame(X_num_testscaled, columns=numeric_cols,index=X_num.index)
xtestscaler1 = pd.concat([X_num_testscaled_df, X_catgorytrain], axis=1)

## Feature Engineering

In [ ]:
# สร้างฟีเจอร์ใหม่จากข้อมูลเดิม							
	วันเกิด → อายุ							
	ทำ binning เช่น อายุแบ่งเป็นช่วง (0–18, 19–35, 36–60, 60+) 							
	เป็น category กลุ่มอายุแต่ละ Gen							
	วันที่ซื้อสินค้า → วันในสัปดาห์, เดือน, ฤดูกาล			จันทร์ อังคาร พุธ พฤหัส ศุกร์ เสาร์ อาทิตย์				เดือน
	ที่อยู่ → รหัสไปรษณีย์, ภูมิภาค							
#  แปลงข้อมูลให้อยู่ในรูปที่โมเดลใช้ได้							
	ข้อความ → TF-IDF, Word Embedding			ใน NLP อันนี้ไม่ต้อง				
	หมวดหมู่ → One-hot encoding หรือ Label encoding							
#   รวมฟีเจอร์เพื่อสร้างข้อมูลเชิงลึก							
	รายได้ต่อเดือน ÷ จำนวนสมาชิกครอบครัว → รายได้ต่อหัว							
	ยอดขาย ÷ จำนวนวัน → ยอดขายเฉลี่ยต่อวัน							
#   ลด noise							
	ตัดค่า Outlier ที่ผิดปกติจริงๆออก มันเกิด noise							


## Feature Selection

In [ ]:
## ใช้วิธี Correlation ดูเบื้องต้น แต่สรุปไม่ได้ว่า feature ไหนสำคัญกับ model

# สมมติว่า X_train เป็น DataFrame และ y_train เป็น Series ของ target
# รวม X_train และ y_train เข้าด้วยกันเพื่อคำนวณ correlation

from sklearn.preprocessing import LabelEncoder

dfcor = pd.concat([X_trainTreebase,ytrainall], axis=1)
dfcor.tail()
corr_matrix = dfcor.corr()
print(corr_matrix)  



In [ ]:
## ใช้วิธี Embedding โยนเข้า Xgboost  เพื่อหา feature importance
## กลุ่ม Treebase model ใช้ x train y train ที่ไม่ปรับ scale

# รอบแรก: train เพื่อดู feature importance
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=200, random_state=42)

model.fit(X_trainTreebase,ytrainall)

# ดู feature importance
importances = model.feature_importances_
feature_names = X_trainTreebase.columns
important_features = [f for f, imp in zip(feature_names, importances) if imp > 0.01]

print(pd.DataFrame(important_features))



#   Train Model จริง

In [ ]:
# รอบสอง: train จริงด้วยเฉพาะ important features
# ทดลอง 3 -4 Model

from xgboost import XGBClassifier

X_trainTree = X_trainTreebase[important_features]
X_testTree = X_testTreebase[important_features]

X_trainDis = xtrainscaler1[important_features]
X_testDis = xtrainscaler1[important_features]


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
## กลุ่ม Tree base model  
# 1.XGBoost
modelxgb = XGBClassifier(n_estimators=500, random_state=42)
modelxgb.fit(X_trainTree,ytrainall)

# 2.Decesion Tree


# 3.Random Forrest



# 4.LightGbm




In [ ]:
#  กลุ่ม Distance Model
# 6.K-Nerest Neighbors



# 7.Support Vector Machine

In [ ]:
# ลองเทียบ ว่า ระหว่างเอา feature x เข้าทั้งหมดกับ feature importance อันไหนดีกว่า
# xtestscaler1 คือตัวเต็มที่ปรับ scaler แล้ว  model ธรรมดา
# X_test_sel คือตัวที่เลือกเฉพาะ feature importance จากการ train รอบแรก model_final

y_pred1 = modelxgb.predict(X_testTree)
y_pred2 = 
# y_predall = model.predict(X_testTree)


In [99]:
from sklearn.metrics import classification_report
print(classification_report(ytestall, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.81      0.85      2848
           1       0.85      0.81      0.83       824
           2       0.89      0.85      0.87      5210
           3       0.97      0.95      0.96      6153
           4       0.92      0.95      0.94     84991
           5       0.92      0.93      0.93     10726
           6       0.93      0.91      0.92     63552

    accuracy                           0.93    174304
   macro avg       0.91      0.89      0.90    174304
weighted avg       0.93      0.93      0.93    174304

